# 5G Network Intelligence — Chronos-2 Multivariate Fine-Tuning

## Research question and method

Can LoRA adaptation of a synthetic-pretrained foundation model substantially improve original DLPRB forecasting? This replaces further small blend searches. The earlier Chronos-Bolt experiment was univariate zero-shot; it did not test this checkpoint or native past-only KPI covariates.


## Temporal protocol

Fit/stop/score cutoffs: 335/383/443 and 395/443/503. Context 168 hours; horizons +1/+6/+24h. All original beams. The cloud bundle redacts original values from 504 onward. No holdout use or automatic production promotion. Reused development results cannot establish fresh-test generalization.


In [ ]:
# Repository review never fits or selects a checkpoint on Run All.
RUN_TRAINING = False


## Configuration

Pinned model revision and weight/config checksums; Chronos 2.3.2, Transformers 4.57.6 and PEFT 0.18.1. LoRA rank 8, alpha 16, learning rate 1e-5, 1,000 steps per fold, seed 17. Batch size 256 counts target and covariate variates, not 256 independent four-channel beam tasks.


In [ ]:
from pathlib import Path
import json
import yaml
import pandas as pd
from IPython.display import display
PROJECT = Path.cwd()
config = PROJECT / 'configs/chronos2_lora.yaml'
if config.exists():
    protocol = yaml.safe_load(config.read_text(encoding='utf-8'))
    display(pd.DataFrame(protocol['folds']))


## Source provenance

Model: `autogluon/chronos-2-synth`, Apache-2.0. The publisher states synthetic-only pretraining; this is not an independent audit of its whole corpus. Downloaded safetensors and config are separately verified. No remote custom code is loaded. Primary sources and method comparison are in `docs/foundation_finetuning_research.md`.


## Causal preprocessing

Each beam supplies DLPRB as the target and DLThpVol, DLThpTime and MR_number as three past-only covariates. No future actual covariates, beam summaries or cross-beam batch learning. Preserve observed zeros and native NaN masks. Fine-tuning receives only each fold's fit prefix. Minimum past is 168 and prediction length 24, so sampled supervised windows cannot cross the fit cutoff. Native scaling is computed from available context.


## Training controls and debugging

Hosted CUDA only, with checkpoints on Drive or Kaggle Output. A callback rejects missing PEFT or an unexpectedly large trainable fraction before optimization. Save optimizer/RNG state every 100 steps; do not silently restart existing output directories. Logs show fold, cutoff, parameter counts and step losses. Fixed-budget training completion is not checkpoint-selection or quality completion.


In [ ]:
if RUN_TRAINING:
    import subprocess, sys
    (PROJECT / 'reports').mkdir(exist_ok=True)
    command = [sys.executable, '-u', '-m', 'scripts.train_chronos2_lora', '--model-root', str(MODEL_ROOT)]
    with (PROJECT / 'reports/chronos2_lora_training.log').open('w') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        TRAIN_EXIT = process.wait()
else:
    print('Review mode: fine-tuning skipped.')


## Checkpoint selection and evaluation — separate required stage

Candidates are steps 100/300/1000, selected by original-unit DLPRB stopping MAE, ties earlier. Native quantile-loss checkpoint selection is disabled. Compare the selected adapter with the same-checkpoint multivariate zero-shot control, verified hurdle and MLP-B/hurdle blend. Report all frozen scoring origins and beams, MAE/WAPE, zero errors, seed variability and independent inference parity. No quality gain follows solely from completing training.


In [ ]:
# Review existing training metadata; no fitting or checkpoint search.
for path in sorted((PROJECT / 'artifacts/chronos2_lora').glob('*/training_protocol.json')):
    print(path.parent.name)
    display(json.loads(path.read_text()))


## Limitations

One initial seed and reused development data. Pretrained representations and more data do not guarantee improvement. The first stage isolates target fine-tuning; external-domain adaptation needs its own matched-compute controls. Preserve mixed/negative results. WAPE <10% remains unmet until verified forecasting evidence proves otherwise.
